**Feature engineering on the preprocessed CSIC 2010 dataset**

Convert request-level records into a numerical feature matrix: drop non-predictive identifiers (`cookie`, `index`), one-hot encode categorical fields (`method`, `host`), extract evidence-based URL/payload/length features, and define the binary target `anomalous`. Store the result as `csic_2010_featurised.parquet` for model training and evaluation.

In [55]:
import os
import re
import random
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [56]:
# working directory for relative paths
notebooks_path = Path.cwd() # <repo-root>/noteboooks
repo_root_path = notebooks_path.parent # <repo-root>/
data_path = repo_root_path / "data" # <repo-root>/data
input_dataset_path = data_path / "csic_2010_preprocessed.parquet" # input: preprocessed dataset 
output_dataset_path = data_path / "csic_2010_featurised.paraquet" # output: featurised dataset

**Load the preprocessed dataset**

Read `csic_2010_preprocessed.parquet` — the request-level dataset (one row per HTTP request) produced by `preprocess.ipynb` — and confirm its shape and columns before feature engineering.   

In [58]:
df = pd.read_parquet(input_dataset_path)
print(f"loaded: {input_path.relative_to(repo_root_path)}")
print(f"shape: {df.shape}")
df.head()

loaded: data/csic_2010_preprocessed.csv
shape: (61065, 8)


,cookie,index,method,url,host,label,payload,contentLength
0,JSESSIONID=0000442E26C2F96F52C97120FF00254B,6883,GET,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,NaN
1,JSESSIONID=0000AF0F87A92C3DFEE827BDFC243736,4687,GET,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,NaN
2,JSESSIONID=00020FC2706BF9F82BE6D4A7BBE12F41,25623,GET,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,NaN
3,JSESSIONID=0002AB6A7AC61AE9DC71925807460D8C,10367,GET,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,NaN
4,JSESSIONID=0003D3946ADEBD8EE20F3B4700193661,19522,POST,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   cookie         61065 non-null  object 
 1   index          61065 non-null  int64  
 2   method         61065 non-null  object 
 3   url            61065 non-null  object 
 4   host           61065 non-null  object 
 5   label          61065 non-null  object 
 6   payload        61065 non-null  object 
 7   contentLength  17977 non-null  float64
dtypes: float64(1), int64(1), object(6)
memory usage: 3.7+ MB


**Observations:**
- `61,065` HTTP requests, 8 columns (`cookie`, `index`, `method`, `url`, `host`, `label`, `payload`, `contentLength`).
- All columns are fully populated except `contentLength` `17,977` non-null,  structurally missing for GET requests, as verified in preprocessing.

**Handle missing `contentLength`**

Preprocessing established that `contentLength` is NaN only for GET requests (no body) and non-null for all POST/PUT requests. Fill with 0 and cast to integer.

In [61]:
n_missing = df["contentLength"].isna().sum()
df["contentLength"] = df["contentLength"].fillna(0).astype(int)
print(f"contentLength: filled {n_missing} missing values with 0, cast to int")
df.info()

contentLength: filled 43088 missing values with 0, cast to int
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   cookie         61065 non-null  object
 1   index          61065 non-null  int64 
 2   method         61065 non-null  object
 3   url            61065 non-null  object
 4   host           61065 non-null  object
 5   label          61065 non-null  object
 6   payload        61065 non-null  object
 7   contentLength  61065 non-null  int64 
dtypes: int64(2), object(6)
memory usage: 3.7+ MB


**Analyse column values to derive features**

Before deriving features, assess each column's predictive value. Identifiers and sequence artifacts must be dropped (they invite memorization, not generalization); informative columns get transformed into numerical features. Decisions are recorded per column, with evidence.

In [64]:
random.seed(42)   # reproducible sampling in the sections below

print(f"cookie:        {df['cookie'].nunique()}")
print(f"index:         {df['index'].nunique()}")
print(f"method:        {df['method'].nunique()}")
print(f"url:           {df['url'].nunique()}")
print(f"host:          {df['host'].nunique()}")
print(f"label:         {df['label'].nunique()}")
print(f"payload:       {df['payload'].nunique()}")
print(f"contentLength: {df['contentLength'].nunique()}")

cookie:        61065
index:         36000
method:        3
url:           1643
host:          2
label:         2
payload:       12093
contentLength: 383


**Observations:**  
- `index`: It is a request-sequence artifact, reused across sessions. No predictive semantics; drop.
- `cookie`: It is a unique per-request identifier (`61,065` unique values, verified in preprocessing). Including it would let the model memorize training requests with zero generalization; drop (retained in preprocessed data for traceability).
- `method`, `host`: These have low-cardinality categoricals; one-hot encode
- `url`, `payload`: These are free-text fields; derive pattern/length features
- `contentLength`: This is numeric with derived flags; keep as-is with derived features
- `label`: The target; Binary encode as `anomalous`

In [74]:
df_featurised = df.drop(columns=["index"])
print(f"Dropped: ['index']\nRemaining: {df_featurised.columns.tolist()}")

Dropped: ['index']
Remaining: ['cookie', 'method', 'url', 'host', 'label', 'payload', 'contentLength']


**Analyse `cookie` column values**

Confirm that cookie values are unique per request and structurally uniform i.e. the column is an identifier, not a signal.

In [75]:
# Random cookie samples
random_row_numbers = random.sample(range(0, 61066), 5)
for rn in random_row_numbers:
    print(f"row#{rn}: {df_featurised.iloc[rn]['cookie']}")

row#41905: JSESSIONID=AF557CC6DB07CEAE226CC829394A0684
row#7296: JSESSIONID=1E0F8DAC192110B7000725D51861BF68
row#1639: JSESSIONID=06C8D233603F71CF42E1657261C37896
row#48598: JSESSIONID=CB206376C064CD79CBD20A6EA1C75BFE
row#18024: JSESSIONID=4AF02A9587C5E1ABFF70FEEE5C84FB77


In [76]:
cookie_lengths = df_featurised["cookie"].str.len()
print(f"unique cookies: {df_featurised['cookie'].nunique()} / {len(df_featurised)}")
print(f"length: min={cookie_lengths.min()}, max={cookie_lengths.max()}, mean={cookie_lengths.mean():.2f}")

valid_count = df_featurised["cookie"].str.match(r"^JSESSIONID=[A-Z0-9]+$").sum()
print(f"valid JSESSIONID format: {valid_count}")

unique cookies: 61065 / 61065
length: min=43, max=43, mean=43.00
valid JSESSIONID format: 61065


**Observations:**  
Every `cookie` value is unique (`61,065` / `61,065`) with a fixed `43-char` `JSESSIONID=HEX32` format. The column is a pure identifier - maximal value variability (unique per request) with zero generalizable structure, including it would enable memorization of training requests. Drop 

In [77]:
df_featurised = df_featurised.drop(columns=["cookie"])
print(f"Dropped: ['cookie']\nRemaining: {df_featurised.columns.tolist()}")

Dropped: ['cookie']
Remaining: ['method', 'url', 'host', 'label', 'payload', 'contentLength']


**Analyse `Method` column values**

The HTTP method indicates the request type. `POST`/`PUT` requests carry bodies (payloads) and may exhibit different anomaly characteristics than `GET`. Inspect the distribution, then one-hot encode.

In [78]:
print(df_featurised['method'].value_counts())

method
GET     43088
POST    17580
PUT       397
Name: count, dtype: int64


**Observations:**  
`GET` dominates (`43,088` = `70.6%`), followed by `POST` (`17,580` = `28.8%`) and `PUT` (`397` = `0.6%`). Recall from preprocessing that `contentLength` presence is fully determined by method (`GET` = `NaN`, `POST`/`PUT` = present), so the method encoding also carries the has-body signal. As a categorical with no ordinal relationship, encode as one-hot.

In [79]:
# One-hot encode `method`
method_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
method_cols = ["method"]

method_encoder.fit(df_featurised[method_cols])
method_encoded_cols = list(method_encoder.get_feature_names_out(method_cols))
print(f"categories:  {method_encoder.categories_}")
print(f"new columns: {method_encoded_cols}")

df_featurised[method_encoded_cols] = method_encoder.transform(df_featurised[method_cols])
df_featurised = df_featurised.drop(columns=method_cols)
print(f"columns: {df_featurised.columns.tolist()}")

categories:  [array(['GET', 'POST', 'PUT'], dtype=object)]
new columns: ['method_GET', 'method_POST', 'method_PUT']
columns: ['url', 'host', 'label', 'payload', 'contentLength', 'method_GET', 'method_POST', 'method_PUT']


`method` is replaced by three binary features `method_GET`, `method_POST`, `method_PUT`. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

### 3.3 Analyse `host` column values

In [16]:
df_featurised['host'].value_counts()

host
localhost:8080    60668
localhost:9090      397
Name: count, dtype: int64

**Observations:**  
- Identifies the `server:port` for the request.
- Contains two host values `localhost:8080` and `localhost:9090`  
- Majority are `localhost:8080` `60,668` following by `localhost:9090` `397`
- Useful for filtering or grouping requests by host.
- `host` column is highly imbalanced, which may limit its predictive power.

In [17]:
# Initialize the encoder
host_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
category_col = ["host"]

# Fit the encoder to the data
host_encoder.fit(df_featurised[category_col])
print(f"Encoder categories:\n\t{host_encoder.categories_}")

# Generate column names for the encoded features
host_encoded_cols = list(host_encoder.get_feature_names_out(category_col))
print(f"Generated column names:\n\t{host_encoded_cols}")

Encoder categories:
	[array(['localhost:8080', 'localhost:9090'], dtype=object)]
Generated column names:
	['host_localhost:8080', 'host_localhost:9090']


The host column is transformed into two binary features - `host_localhost:8080` and `host_localhost:9090`. This allows the model to interpret the request type as numerical input while preserving the categorical relationships.

Apply the encoder to the dataset and concatenate the encoded features with the original DataFrame for modeling.

In [18]:
df_featurised[host_encoded_cols] = host_encoder.transform(df_featurised[category_col]).copy()
df_featurised.head(5)

,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090
0,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,NaN,1.0,0.0,0.0,1.0,0.0
1,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,NaN,1.0,0.0,0.0,1.0,0.0
2,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,NaN,1.0,0.0,0.0,1.0,0.0
3,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,NaN,1.0,0.0,0.0,1.0,0.0
4,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0


### 3.4 Analyse URL column values

Count unique URLs in the dataset

In [19]:
print(f"Unique URLs: {df_featurised['url'].nunique()}")

Unique URLs: 1643


Sample a few random URLs for analysis

In [20]:
random_row_numbers = random.sample(range(0, 61066), 10)
for rn in random_row_numbers:
    print(f"row[{rn}]: {df_featurised.iloc[rn]['url']}")

row[1158]: http://localhost:8080/tienda1/imagenes/3.gif/4861362529278789730~
row[32714]: http://localhost:8080/tienda1/publico/anadir.jsp
row[52088]: http://localhost:8080/tienda1/global/creditos.jsp
row[32729]: http://localhost:8080/tienda1/imagenes/nuestratierra.jpg
row[53011]: http://localhost:8080/tienda1/publico/pagar.jsp
row[45650]: http://localhost:8080/tienda1/publico/anadir.jsp
row[9517]: http://localhost:8080/tienda1/miembros/editar.jsp
row[24917]: http://localhost:8080/tienda1/miembros/editar.jsp
row[34875]: http://localhost:8080/tienda1/miembros/editar.jsp
row[1724]: http://localhost:8080/tienda1/imagenes/nuestratierra.jpg


**Observations:**  
URLs contain patterns, structures, and metadata that can be leveraged for feature engineering.  
Eg:
- url length
- Count of special chars: `/` `.` `?` `&` etc
- Presence of query params
- Presence of suspicious patterns: sql select, union, drop etc
- Presence of path traversal

Analyse `urls` in detail for presence of `special chars`, `patterns` etc that are `label`'ed  anomalous

In [51]:
from collections import defaultdict

url_df = df_featurised[["url", "label"]]

# Special characters
special_chars = ['/', '.', '?', '&', '=', '%']
total_urls_per_special_char = defaultdict(int)

# SQL patterns
sql_patterns = [ r"select", r"union", r"drop", r"insert\s+into", r"delete"]
total_urls_per_sql_pattern = defaultdict(int)
for pattern in sql_patterns:
    total_urls_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_urls_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_urls_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_urls_with_or_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_urls_with_comment_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_urls_with_path_traversal_pattern = 0

# Parameter pattern
parameter_regex_pattern = r"[?&][^=&]+="
total_urls_with_parameter_pattern = 0

# Trailing digits
trailing_digits_regex_pattern = r'http://localhost:(?:8080|9090).*?(\d+)'
total_urls_with_trailing_digits_pattern = 0

# Numeric ID
numeric_id_regex_pattern = r"/\d{10,}"
total_urls_with_numeric_id_pattern = 0

# Track label distribution for each pattern
label_distribution = {
    "select": {"anom": 0, "norm": 0},
    "drop": {"anom": 0, "norm": 0},
    "script": {"anom": 0, "norm": 0},
    "alert": {"anom": 0, "norm": 0},
    "or": {"anom": 0, "norm": 0},
    "comment": {"anom": 0, "norm": 0},
    "path_traversal": {"anom": 0, "norm": 0},
    "parameter": {"anom": 0, "norm": 0},
    "trailing_digits": {"anom": 0, "norm": 0},
    "numeric_id": {"anom": 0, "norm": 0},
}

# Iterate through each URL and its corresponding label
for url, label in zip(url_df["url"], url_df["label"]):

    # Count special characters
    for char in special_chars:
        count = url.count(char)
        if count > 0:
            total_urls_per_special_char[char] += 1
        else:
            total_urls_per_special_char[char] += 0

    for pattern in sql_patterns:
        if re.search(pattern, url):
            total_urls_per_sql_pattern[pattern] +=  1
            label_distribution[pattern][label] += 1

    if re.search(script_regex_pattern, url):
        total_urls_with_script_pattern += 1
        label_distribution["script"][label] += 1
        
    if re.search(alert_regex_pattern, url):
        total_urls_with_alert_pattern += 1
        label_distribution["alert"][label] += 1

    if re.search(or_regex_pattern, url):
        total_urls_with_or_pattern += 1
        label_distribution["or"][label] += 1
        
    if re.search(comment_regex_pattern, url):
        total_urls_with_comment_pattern += 1
        label_distribution["comment"][label] += 1
        
    if re.search(path_traversal_regex_pattern, url):
        total_urls_with_path_traversal_pattern += 1
        label_distribution["path_traversal"][label] += 1

    if re.search(parameter_regex_pattern, url):
        total_urls_with_parameter_pattern += 1
        label_distribution["parameter"][label] += 1
    
    if re.search(trailing_digits_regex_pattern, url):
        total_urls_with_trailing_digits_pattern += 1
        label_distribution["trailing_digits"][label] += 1
        
    if re.search(numeric_id_regex_pattern, url):
        total_urls_with_numeric_id_pattern += 1
        label_distribution["numeric_id"][label] += 1
    
# Display the total count of URLs containing each special character
print("Total URLs containing per special Character:")
for char, count in total_urls_per_special_char.items():
    print(f"    {char}: {count} URLs")

print("Total URLs containing per SQL pattern:")
for pattern, count in total_urls_per_sql_pattern.items():
    print(f"    {pattern}: {count} URLs")

print(f"Total URLs containing other patterns:")
print(f"   'or': {total_urls_with_or_pattern}")
print(f"   'script': {total_urls_with_script_pattern}")
print(f"   'alert': {total_urls_with_alert_pattern}")
print(f"   'comment': {total_urls_with_comment_pattern}")
print(f"   'path traversal': {total_urls_with_path_traversal_pattern}")
print(f"   'parameter': {total_urls_with_parameter_pattern}")
print(f"   'trailing digits': {total_urls_with_trailing_digits_pattern}")
print(f"   'numeric id': {total_urls_with_numeric_id_pattern}")

# Display label distribution for each pattern
print("\nLabel distribution for patterns:")
for pattern, counts in label_distribution.items():
    print(f"   {pattern}: {counts}")


Total URLs containing per special Character:
    /: 61065 URLs
    .: 60386 URLs
    ?: 0 URLs
    &: 0 URLs
    =: 0 URLs
    %: 0 URLs
Total URLs containing per SQL pattern:
    select: 8 URLs
    union: 0 URLs
    drop: 5 URLs
    insert\s+into: 0 URLs
    delete: 0 URLs
Total URLs containing other patterns:
   'or': 0
   'script': 0
   'alert': 0
   'comment': 0
   'path traversal': 0
   'parameter': 0
   'trailing digits': 59329
   'numeric id': 967

Label distribution for patterns:
   select: {'anom': 8, 'norm': 0}
   drop: {'anom': 5, 'norm': 0}
   script: {'anom': 0, 'norm': 0}
   alert: {'anom': 0, 'norm': 0}
   or: {'anom': 0, 'norm': 0}
   comment: {'anom': 0, 'norm': 0}
   path_traversal: {'anom': 0, 'norm': 0}
   parameter: {'anom': 0, 'norm': 0}
   trailing_digits: {'anom': 23329, 'norm': 36000}
   numeric_id: {'anom': 967, 'norm': 0}


**Observations:**

- Special Characters:  
  - `/` and `.` are too common and do not help distinguish anomalies.  
  - `?`, `&`, `=`, `%` do not appear in the dataset, so they provide no signal.  
  > **Conclusion:** Exclude all special character counts from the model

- SQL Patterns:  
  - `select` and `drop` are rare but meaningful for detecting SQL injection.  
  - `union`, `insert`, `delete` do not appear in the dataset.
  > **Conclusion:** Include `select` and `drop` as binary features to for detecting SQL injection attempts.

- Other Patterns:  
  - All patterns `or`, `script`, `alert`, `comment`, `path traversal`, `parameter` have zero occurrences in the dataset.  
  - `Trailing digits` appears in both anom and norm URLs and is not discriminative
  - `Numeric ID` is a strong indicator of anomalies. All of its occurrences are in anom URLs.
  > **Conclusion:** Include `numeric_id` as a binary feature. Exclude all other patterns from the model.

Add the `url` derived feature columns to dataframe

In [22]:
df_featurised["url_has_select"] = df_featurised["url"].str.contains("select", case=False).astype(int)
df_featurised["url_has_drop"] = df_featurised["url"].str.contains("drop", case=False).astype(int)
df_featurised["url_has_numeric_id"] = df_featurised["url"].apply(lambda x: 1 if re.search(r"/\d{10,}", x) else 0).astype(int)

# Additional features
df_featurised["url_length"] = df_featurised["url"].apply(len)

# Display the new features
#df_featurised[["url", "label", "url_has_select", "url_has_drop", "url_has_numeric_id", "url_length"]].head()
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   url                  61065 non-null  object 
 1   host                 61065 non-null  object 
 2   label                61065 non-null  object 
 3   payload              61065 non-null  object 
 4   contentLength        17977 non-null  float64
 5   method_GET           61065 non-null  float64
 6   method_POST          61065 non-null  float64
 7   method_PUT           61065 non-null  float64
 8   host_localhost:8080  61065 non-null  float64
 9   host_localhost:9090  61065 non-null  float64
 10  url_has_select       61065 non-null  int64  
 11  url_has_drop         61065 non-null  int64  
 12  url_has_numeric_id   61065 non-null  int64  
 13  url_length           61065 non-null  int64  
dtypes: float64(6), int64(4), object(4)
memory usage: 6.5+ MB


### 3.5 Analyse `payload` column values

Count unique `payloads` in the dataset

In [23]:
print(f"Unique Payloads: {df_featurised['payload'].nunique()}")

Unique Payloads: 12093


Sample a few random Payloads for analysis

In [46]:
unique_values = df_featurised['payload'].unique()
random.seed(42)
random_row_numbers = random.sample(range(0, len(unique_values)), 10)
for rn in random_row_numbers:
    print(f"    {unique_values[rn]}\n")

    modo=registro login=suen password=OBED9e3cIA0 nombre=Yazm%EDn apellidos=G%E9lvez+Matons email=locatell%40whitehat.no dni=52438346W direccion=Calle+Guadamacileros%2C+24+6-C ciudad=Real+de+Montroi cp=12190 provincia=Tarragona ntc=4960204900106795 B1=Registrar

    modo=insertar precio=4344 B1=%27%3B+DROP+TABLE+usuarios%3B+SELECT+*+FROM+datos+WHERE+nombre+LIKE+%27%25

    modo=registro login=lenka password=096r13.a nombre=Daher apellidos=De+la+Ossa email=heathcock%40catanoticias.ai dni=30598769Y direccion=Avinguda+Josep+Tarradellas%2C+146+11%3FH ciudadA=Asparrena cp=09614 provincia=Salamanca ntc=9256871271036534 B1=Registrar

    errorMsg=Credenciales+incorrectas%27+DELETE+FROM+USERS

    id=2 nombre=Ques%2Bo+Manchego precio=39 cantidad=88 B1=A%F1adir+al+carrito

    id=2 nombre=Vino+Rioja%252B precio=100 cantidad=86 B1=A%F1adir+al+carrito

    modo=registro login=marymarg password=m1r66m7%24o nombre=Barbarina apellidos=Hodzic+Fajardo email=bibb%40yksoluap.io dni=05554476E direccionA=

**Observations:**  
- Payloads show form submissions, login attempts, data entries etc 
- Typically formatted as `key-value` pairs (e.g., modo=registro, login=quek, password=5a69io99t3).
- Feature can be derived based on structure, encoded chars, sql keywords, long numeric string, suspicious patterns etc

Analyse payloads in detail for presence of special chars, patterns etc that are label'ed anomalous

In [43]:
from collections import defaultdict

payload_df = df_featurised[["payload", "label"]]

# Special characters
special_chars = ['&', '=', '%']
total_payloads_per_special_char = defaultdict(int)

# Count digits
digits_regex_pattern= r"\d"
total_payloads_with_digits_pattern = 0

# Parameter pattern
#parameter_regex_pattern = r"[?&][^=&]+="
parameter_regex_pattern = r"(^|&)[^=&]+="
total_payloads_with_parameter_pattern = 0

# Quote pattern
quote_regex_pattern = r"'|%27|\"|%22"
total_payloads_with_quote_pattern = 0

# Angle bracket pattern
angle_bkt_regex_pattern = r"<|>|%3c|%3e"
total_payloads_with_angle_bkt_pattern = 0

# parenthesis pattern
parenthesis_regex_pattern = r"\(|\)|%28|%29"
total_payloads_with_parenthesis_pattern = 0

# Semicolon pattern
semicolon_regex_pattern = r";|%3b"
total_payloads_with_semicolon_pattern = 0

# Slash pattern
slash_regex_pattern = r"/|%2f"
total_payloads_with_slash_pattern = 0

# Comment pattern
comment_regex_pattern = r"--|/\*|\*/|%2d%2d"
total_payloads_with_comment_pattern = 0

# Percent encoded pattern
percent_encoded_regex_pattern = r"%[0-9a-fA-F]{2}"
total_payloads_with_percent_encoded_pattern = 0

# SQL patterns
sql_patterns = [ r"select", r"union", r"drop", r"insert\s+into", r"delete"]
total_payloads_per_sql_pattern = defaultdict(int)
for pattern in sql_patterns:
    total_payloads_per_sql_pattern[pattern] = 0

# Script Pattern
script_regex_pattern = r"<script|%3cscript"
total_payloads_with_script_pattern = 0

# Alert Pattern
alert_regex_pattern = r"alert|%61%6c%65%72%74"
total_payloads_with_alert_pattern = 0

# OR pattern
or_regex_pattern = r"\bor\b|\|\|"
total_payloads_with_or_pattern = 0

# Path traversal pattern
path_traversal_regex_pattern = r"\.\./|%2e%2e%2f"
total_payloads_with_path_traversal_pattern = 0


# Track label distribution for each pattern
payload_label_distribution = {
    "digits": {"anom": 0, "norm": 0},
    "parameter": {"anom": 0, "norm": 0},    
    "quote": {"anom": 0, "norm": 0},
    "angle_bkt": {"anom": 0, "norm": 0},
    "parenthesis": {"anom": 0, "norm": 0},
    "semicolon": {"anom": 0, "norm": 0},
    "slash": {"anom": 0, "norm": 0},
    "comment": {"anom": 0, "norm": 0},
    "percent_encoded": {"anom": 0, "norm": 0},
    
    "select": {"anom": 0, "norm": 0},
    "union": {"anom": 0, "norm": 0},
    "drop": {"anom": 0, "norm": 0},
    "insert\\s+into": {"anom": 0, "norm": 0},
    "delete": {"anom": 0, "norm": 0},
 
    "script": {"anom": 0, "norm": 0},
    "alert": {"anom": 0, "norm": 0},
    "or": {"anom": 0, "norm": 0},
    "path_traversal": {"anom": 0, "norm": 0},    
}

# Iterate through each Payload and its corresponding label
for payload, label in zip(payload_df["payload"], payload_df["label"]):

    # Count special characters
    for char in special_chars:
        count = payload.count(char)
        if count > 0:
            total_payloads_per_special_char[char] += 1
        else:
            total_payloads_per_special_char[char] += 0

    if re.search(digits_regex_pattern, payload):
        total_payloads_with_digits_pattern += 1
        payload_label_distribution["digits"][label] += 1

    if re.search(parameter_regex_pattern, payload):
        total_payloads_with_parameter_pattern += 1
        payload_label_distribution["parameter"][label] += 1

    if re.search(quote_regex_pattern, payload):
        total_payloads_with_quote_pattern += 1
        payload_label_distribution["quote"][label] += 1
    
    if re.search(angle_bkt_regex_pattern, payload):
        total_payloads_with_angle_bkt_pattern += 1
        payload_label_distribution["angle_bkt"][label] += 1

    if re.search(parenthesis_regex_pattern, payload):
        total_payloads_with_parenthesis_pattern += 1
        payload_label_distribution["parenthesis"][label] += 1

    if re.search(semicolon_regex_pattern, payload):
        total_payloads_with_semicolon_pattern += 1
        payload_label_distribution["semicolon"][label] += 1
    
    if re.search(slash_regex_pattern, payload):
        total_payloads_with_slash_pattern += 1
        payload_label_distribution["slash"][label] += 1    

    if re.search(percent_encoded_regex_pattern, payload):
        total_payloads_with_percent_encoded_pattern += 1
        payload_label_distribution["percent_encoded"][label] += 1

    for pattern in sql_patterns:
        if re.search(pattern, payload):
            total_payloads_per_sql_pattern[pattern] +=  1
            payload_label_distribution[pattern][label] += 1
            
    if re.search(script_regex_pattern, payload):
        total_payloads_with_script_pattern += 1
        payload_label_distribution["script"][label] += 1
        
    if re.search(alert_regex_pattern, payload):
        total_payloads_with_alert_pattern += 1
        payload_label_distribution["alert"][label] += 1

    if re.search(or_regex_pattern, payload):
        total_payloads_with_or_pattern += 1
        payload_label_distribution["or"][label] += 1
        

    if re.search(path_traversal_regex_pattern, payload):
        total_payloads_with_path_traversal_pattern += 1
        payload_label_distribution["path_traversal"][label] += 1

# Display the total count of payloads containing each special character
print("Total payloads containing per special Character:")
for char, count in total_payloads_per_special_char.items():
    print(f"    {char}: {count} payloads")

print("Total payloads containing per SQL pattern:")
for pattern, count in total_payloads_per_sql_pattern.items():
    print(f"    {pattern}: {count} payloads")

print(f"Total payloads containing other patterns:")
print(f"   'parameter': {total_payloads_with_parameter_pattern}")
print(f"   'digits': {total_payloads_with_digits_pattern}")
print(f"   'quote': {total_payloads_with_quote_pattern}")
print(f"   'angle_bkt': {total_payloads_with_angle_bkt_pattern}")
print(f"   'parenthesis': {total_payloads_with_parenthesis_pattern}")
print(f"   'semicolon': {total_payloads_with_semicolon_pattern}")
print(f"   'slash': {total_payloads_with_slash_pattern}")
print(f"   'comment': {total_payloads_with_comment_pattern}")
print(f"   'percent_encoded': {total_payloads_with_percent_encoded_pattern}")
print(f"   'script': {total_payloads_with_script_pattern}")
print(f"   'alert': {total_payloads_with_alert_pattern}")
print(f"   'or': {total_payloads_with_or_pattern}")
print(f"   'path traversal': {total_payloads_with_path_traversal_pattern}")


# Display label distribution for each pattern
print("\nLabel distribution for patterns:")
for pattern, counts in payload_label_distribution.items():
    print(f"   {pattern}: {counts}")


Total payloads containing per special Character:
    &: 0 payloads
    =: 35574 payloads
    %: 21395 payloads
Total payloads containing per SQL pattern:
    select: 0 payloads
    union: 0 payloads
    drop: 4 payloads
    insert\s+into: 0 payloads
    delete: 0 payloads
Total payloads containing other patterns:
   'parameter': 35574
   'digits': 32629
   'quote': 4188
   'angle_bkt': 0
   'paranthesis': 1554
   'semicolon': 0
   'slash': 17
   'comment': 0
   'percent_encoded': 21395
   'script': 0
   'alert': 1158
   'or': 2
   'path traversal': 0

Label distribution for patterns:
   digits: {'anom': 18629, 'norm': 14000}
   parameter: {'anom': 19574, 'norm': 16000}
   quote: {'anom': 4088, 'norm': 100}
   angle_bkt: {'anom': 0, 'norm': 0}
   paranthesis: {'anom': 1524, 'norm': 30}
   semicolon: {'anom': 0, 'norm': 0}
   slash: {'anom': 17, 'norm': 0}
   comment: {'anom': 0, 'norm': 0}
   percent_encoded: {'anom': 15001, 'norm': 6394}
   select: {'anom': 0, 'norm': 0}
   union: {'an

**Observations:** 
- Special Characters:
  - `&` no occurrences and `=` is too common. Both these are not useful
  - `%` is also common but may be useful when combined with `percent_encoded` pattern  
> **Conclusion:**  Exclude `&` and `=`. Use `%` as a feature

- SQL Patterns:  
  -  No occurrences of `select` `union`, `delete` `insert`
  - `drop` appears only in `norm`, so it is not useful for anomaly detection.  
> **Conclusion:** Exclude all other SQL patterns.  

- Other Patterns:
  - `>90%` of `quote` `parenthesis` `slash` `alert` are anomalous. These are strong candidates features  
  - `>50%` of `Parameter` `percent_encoded` are anomalous. Provides resonable discriminative value
  - Rest all almost no signals for anomalous classification 
> **Conclusion:** Include `quote` `parenthesis` `slash` `alert` `Parameter` `percent_encoded` drop the rest

Add the `payload` derived feature columns in the dataset

In [26]:
# Extract the `Payload` columns series
payload_series = df_featurised["payload"].astype(str)

# Count based features
df_featurised["payload_quote_count"] = payload_series.str.count(r"'|%27|\"|%22", flags=re.IGNORECASE)
df_featurised["payload_paren_count"] = payload_series.str.count(r"\(|\)|%28|%29", flags=re.IGNORECASE)
df_featurised["payload_slash_count"] = payload_series.str.count(r"/|%2f", flags=re.IGNORECASE)
df_featurised["payload_parameter_count"] = payload_series.str.count(r"(^|&)[^=&]+=", flags=re.IGNORECASE)
df_featurised["payload_percent_count"] = payload_series.str.count(r"%", flags=re.IGNORECASE)
df_featurised["payload_percent_encoded_count"] = payload_series.str.count(r"%[0-9a-fA-F]{2}", flags=re.IGNORECASE)

# Binary flag based features
df_featurised["payload_has_alert"] = payload_series.str.contains(r"alert|%61%6c%65%72%74", case=False, regex=True)
#df_featurised["payload_has_sql_insert"] = payload_series.str.contains(r"insert", case=False, regex=True)
#df_featurised["payload_has_sql_insert"] = payload_series.str.contains(r"insert\s+into", case=False, regex=True)

# Additional features
df_featurised["payload_length"] = payload_series.str.len()

df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 23 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   url                            61065 non-null  object 
 1   host                           61065 non-null  object 
 2   label                          61065 non-null  object 
 3   payload                        61065 non-null  object 
 4   contentLength                  17977 non-null  float64
 5   method_GET                     61065 non-null  float64
 6   method_POST                    61065 non-null  float64
 7   method_PUT                     61065 non-null  float64
 8   host_localhost:8080            61065 non-null  float64
 9   host_localhost:9090            61065 non-null  float64
 10  url_has_select                 61065 non-null  int64  
 11  url_has_drop                   61065 non-null  int64  
 12  url_has_numeric_id             61065 non-null 

In [27]:
df_featurised.head(5)

,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,...,url_length,payload_quote_count,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_has_alert,payload_has_sql_insert,payload_length
0,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,NaN,1.0,0.0,0.0,1.0,0.0,...,44,0,0,0,0,0,0,False,False,0
1,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,NaN,1.0,0.0,0.0,1.0,0.0,...,70,0,0,0,0,0,0,False,False,0
2,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,NaN,1.0,0.0,0.0,1.0,0.0,...,57,0,0,0,1,0,0,False,False,4
3,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,NaN,1.0,0.0,0.0,1.0,0.0,...,48,0,0,0,1,4,4,False,False,77
4,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0,...,57,2,0,5,1,18,18,False,False,174


### 3.6 Analyse `contentLength` column values

In [49]:
print(f"Unique values: {df_featurised['contentLength'].nunique()}")
df_featurised["contentLength"] = df_featurised["contentLength"].fillna(0).astype(int)
df_featurised['contentLength'].describe()

Unique values: 383


count    61065.000000
mean        31.936903
std         75.660779
min          0.000000
25%          0.000000
50%          0.000000
75%         17.000000
max        836.000000
Name: contentLength, dtype: float64

**Observations:**  

- `50%` of requests have a contentLength of `0`
- mean `31.94` > median `0.0`, indicating most requests have small or no payloads, but a few have very large payloads.

Inspect possibility of anomaly pattern based on `contentLength` statistical range

In [29]:
content_length_df = df_featurised[["contentLength", "label"]]

In [30]:
# label distribution for each statistical range
content_length_label_distribution = {
    "0_to_10": {"anom": 0, "norm": 0},
    "10_to_50": {"anom": 0, "norm": 0},
    "50_to_200": {"anom": 0, "norm": 0},
    "greater_than_200": {"anom": 0, "norm": 0},
}
for contentLength, label in zip(content_length_df["contentLength"], content_length_df["label"]):
    # Categorize contentLength into bins based on the statistics
    if contentLength > 0 and contentLength <= 10:
        content_length_label_distribution["0_to_10"][label] += 1
    elif contentLength > 10 and contentLength <= 50:
        content_length_label_distribution["10_to_50"][label] += 1
    elif contentLength > 50 and contentLength <= 200:
        content_length_label_distribution["50_to_200"][label] += 1
    else: # >200
        content_length_label_distribution["greater_than_200"][label] += 1

# Display the label distribution for each statistical range
print("Label Distribution for contentLength Ranges:")
for range_key, label_distr in content_length_label_distribution.items():
    print(f"   {range_key}: {label_distr}")

Label Distribution for contentLength Ranges:
   0_to_10: {'anom': 683, 'norm': 1000}
   10_to_50: {'anom': 2469, 'norm': 3000}
   50_to_200: {'anom': 3894, 'norm': 2000}
   greater_than_200: {'anom': 18019, 'norm': 30000}


**Observation:**  
- The ratio of anomaly to normal does not monotonically increase or decrease with contentLength.
- Small payloads (0-10 bytes) and very large payloads (>200 bytes) have lower anomalies percentages (~40%).
- Medium payloads (50-200 bytes) have relatively higher anomaly percentage (~66%).

Inspect possibility of anomaly pattern based on contentLength statistical thresholds

In [31]:
import numpy as np
from collections import defaultdict

# Assuming content_length_df contains 'contentLength' and 'label' columns
content_length_df = df_featurised[["contentLength", "label"]]

# Define the statistical thresholds
stats = {
    "min": 0.0,
    "25%": 0.0,
    "50%": 0.0,
    "75%": 17.0,
    "max": 836.0,
    "mean": 31.936903,
    "std": 75.660779
}

# Initialize a dictionary to store label distribution for each statistical range
label_distribution = {
    "min_to_25%": {"anom": 0, "norm": 0},
    "25%_to_50%": {"anom": 0, "norm": 0},
    "50%_to_75%": {"anom": 0, "norm": 0},
    "75%_to_mean": {"anom": 0, "norm": 0},
    "mean_to_mean+std": {"anom": 0, "norm": 0},
    "mean+std_to_max": {"anom": 0, "norm": 0},
    "zero": {"anom": 0, "norm": 0}
}

# Iterate through each contentLength and its corresponding label
for contentLength, label in zip(content_length_df["contentLength"], content_length_df["label"]):
    # Categorize contentLength into bins based on the statistics
    if contentLength == 0:
        label_distribution["zero"][label] += 1
    elif 0 < contentLength <= stats["25%"]:
        label_distribution["min_to_25%"][label] += 1
    elif stats["25%"] < contentLength <= stats["50%"]:
        label_distribution["25%_to_50%"][label] += 1
    elif stats["50%"] < contentLength <= stats["75%"]:
        label_distribution["50%_to_75%"][label] += 1
    elif stats["75%"] < contentLength <= stats["mean"]:
        label_distribution["75%_to_mean"][label] += 1
    elif stats["mean"] < contentLength <= stats["mean"] + stats["std"]:
        label_distribution["mean_to_mean+std"][label] += 1
    elif contentLength > stats["mean"] + stats["std"]:
        label_distribution["mean+std_to_max"][label] += 1

# Display the label distribution for each statistical range
print("Label Distribution for contentLength Ranges:")
for range_name, counts in label_distribution.items():
    total = counts["anom"] + counts["norm"]
    anom_percent = (counts["anom"] / total * 100) if total > 0 else 0
    norm_percent = (counts["norm"] / total * 100) if total > 0 else 0
    print(f"   {range_name}: {counts} (anom: {anom_percent:.2f}%, norm: {norm_percent:.2f}%)")

Label Distribution for contentLength Ranges:
   min_to_25%: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)
   25%_to_50%: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)
   50%_to_75%: {'anom': 808, 'norm': 2000} (anom: 28.77%, norm: 71.23%)
   75%_to_mean: {'anom': 638, 'norm': 0} (anom: 100.00%, norm: 0.00%)
   mean_to_mean+std: {'anom': 4946, 'norm': 4000} (anom: 55.29%, norm: 44.71%)
   mean+std_to_max: {'anom': 3585, 'norm': 2000} (anom: 64.19%, norm: 35.81%)
   zero: {'anom': 0, 'norm': 0} (anom: 0.00%, norm: 0.00%)


**Observations:**  
- 75%_to_mean (17-32 bytes): 100% of payloads in this range are anomalous. However the absolute number of requests `638` are very less compared to the total.
- mean+std_to_max (>108 bytes): 64.19% of payloads in this range are anomalous.
- mean_to_mean+std: 55.29% of payloads in this range are anomalous.
- Other ranges are lesser anomalous percentages

Based on the analysis conducted on the `contentLength` feature, no distinct or consistent pattern emerged that could reliably distinguish between anomalous and normal payloads. For the current iteration, proceeding with the extracted features`is_17_to_32_bytes` and `is_empty_payload` in addition to the original `contentLength` column which already present in the dataset.

In [32]:
# is_17_to_32_bytes
df_featurised["is_17_to_32_bytes"] = (
    (df_featurised["contentLength"] >= 17) &
    (df_featurised["contentLength"] <= 32)
).astype(int)

# is_empty_payload
# df_featurised["is_empty_payload"] = (
#     df_featurised["contentLength"] == 0
# ).astype(int)
df_featurised["is_empty_payload"] = (df_featurised["payload"].str.len() == 0).astype(int)

# Verify the new features
df_featurised.head()

,url,host,label,payload,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,...,payload_paren_count,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_has_alert,payload_has_sql_insert,payload_length,is_17_to_32_bytes,is_empty_payload
0,http://localhost:8080/tienda1/imagenes/2.gif,localhost:8080,norm,,NaN,1.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,False,False,0,0,1
1,http://localhost:8080/tienda1/publico/vaciar.j...,localhost:8080,anom,,NaN,1.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,False,False,0,0,1
2,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,norm,id=2,NaN,1.0,0.0,0.0,1.0,0.0,...,0,0,1,0,0,False,False,4,0,0
3,http://localhost:8080/tienda1/publico/anadir.jsp,localhost:8080,anom,id=3 nombre=Jam%F3n+Ib%E9rico precio=%2500 can...,NaN,1.0,0.0,0.0,1.0,0.0,...,0,0,1,4,4,False,False,77,0,0
4,http://localhost:8080/tienda1/publico/caracter...,localhost:8080,anom,id=2sessionid%3D12312312%26+username%3D%3Cscri...,174.0,0.0,1.0,0.0,1.0,0.0,...,0,5,1,18,18,False,False,174,0,0


In [33]:
print(df_featurised.shape)
print(df_featurised.columns.tolist())

(61065, 25)
['url', 'host', 'label', 'payload', 'contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_has_alert', 'payload_has_sql_insert', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload']


### 3.7 Analyse `label` column values

In [34]:
df_featurised['label'].value_counts()

label
norm    36000
anom    25065
Name: count, dtype: int64

Create a binary target column `anomalous` with `1` for `anom` and `0` for `norm`

In [35]:
df_featurised["anomalous"] = (df_featurised["label"] == "anom").astype(int)
print(df_featurised.shape)
print(df_featurised.columns.tolist())

(61065, 26)
['url', 'host', 'label', 'payload', 'contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_has_alert', 'payload_has_sql_insert', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload', 'anomalous']


## 3.8 Save the Feature-Engineered DataFrame to disk

Save the final DataFrame (with features) as `csic_2010_engineered.paraquet` for modeling.

In [36]:
df_featurised.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61065 entries, 0 to 61064
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   url                            61065 non-null  object 
 1   host                           61065 non-null  object 
 2   label                          61065 non-null  object 
 3   payload                        61065 non-null  object 
 4   contentLength                  17977 non-null  float64
 5   method_GET                     61065 non-null  float64
 6   method_POST                    61065 non-null  float64
 7   method_PUT                     61065 non-null  float64
 8   host_localhost:8080            61065 non-null  float64
 9   host_localhost:9090            61065 non-null  float64
 10  url_has_select                 61065 non-null  int64  
 11  url_has_drop                   61065 non-null  int64  
 12  url_has_numeric_id             61065 non-null 

In [37]:
numeric_cols = df_featurised.select_dtypes(include=[np.number, bool]).columns.tolist()
print(len(numeric_cols))
print(numeric_cols)
#categorical_cols = train_inputs.select_dtypes('object').columns.tolist()

22
['contentLength', 'method_GET', 'method_POST', 'method_PUT', 'host_localhost:8080', 'host_localhost:9090', 'url_has_select', 'url_has_drop', 'url_has_numeric_id', 'url_length', 'payload_quote_count', 'payload_paren_count', 'payload_slash_count', 'payload_parameter_count', 'payload_percent_count', 'payload_percent_encoded_count', 'payload_has_alert', 'payload_has_sql_insert', 'payload_length', 'is_17_to_32_bytes', 'is_empty_payload', 'anomalous']


In [40]:
df_featurised[numeric_cols]

,contentLength,method_GET,method_POST,method_PUT,host_localhost:8080,host_localhost:9090,url_has_select,url_has_drop,url_has_numeric_id,url_length,...,payload_slash_count,payload_parameter_count,payload_percent_count,payload_percent_encoded_count,payload_has_alert,payload_has_sql_insert,payload_length,is_17_to_32_bytes,is_empty_payload,anomalous
0,NaN,1.0,0.0,0.0,1.0,0.0,0,0,0,44,...,0,0,0,0,False,False,0,0,1,0
1,NaN,1.0,0.0,0.0,1.0,0.0,0,0,0,70,...,0,0,0,0,False,False,0,0,1,1
2,NaN,1.0,0.0,0.0,1.0,0.0,0,0,0,57,...,0,1,0,0,False,False,4,0,0,0
3,NaN,1.0,0.0,0.0,1.0,0.0,0,0,0,48,...,0,1,4,4,False,False,77,0,0,1
4,174.0,0.0,1.0,0.0,1.0,0.0,0,0,0,57,...,5,1,18,18,False,False,174,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61060,33.0,0.0,1.0,0.0,1.0,0.0,0,0,0,48,...,0,1,0,0,False,False,33,0,0,0
61061,248.0,0.0,1.0,0.0,1.0,0.0,0,0,0,49,...,0,1,3,3,False,False,248,0,0,1
61062,NaN,1.0,0.0,0.0,1.0,0.0,0,0,0,52,...,0,1,0,0,False,False,58,0,0,1
61063,38.0,0.0,1.0,0.0,1.0,0.0,0,0,0,47,...,0,1,0,0,False,False,38,0,0,0


In [41]:
df_featurised[numeric_cols].to_parquet(output_dataset_path.with_suffix('.parquet'))
print(f"Saved: {output_dataset_path.with_suffix('.parquet')}")

CPU times: user 8 μs, sys: 4 μs, total: 12 μs
Wall time: 27.9 μs
